In [1]:
# 📌 Objectif : Construire le graphe spatial GCN à partir des superpixels labellisés

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph
from torch_geometric.data import Data
import torch

In [9]:
# === 1. Charger les superpixels labellisés ===
df = pd.read_csv("superpixels_with_labels.csv")

# === 2. Features (X), Labels (y), Coordonnées ===
X = df[["mean_ndwi", "mean_srtm"]].values
y = df["label"].astype(int).values
coords = df[["longitude", "latitude"]].values

# === 3. Normaliser les features ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === 4. Créer le graphe d'adjacence avec k plus proches voisins ===
k = 6  # nombre de voisins
A = kneighbors_graph(coords, k, mode='connectivity', include_self=False)
edges = np.array(A.nonzero()).T  # liste des arêtes (i, j)

# === 5. Conversion en objets PyTorch Geometric ===
x = torch.tensor(X_scaled, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)
edge_index = torch.tensor(edges.T, dtype=torch.long)  # shape: [2, num_edges]

data = Data(x=x, edge_index=edge_index, y=y)

# === 6. Sauvegarde temporaire (optionnelle) ===
torch.save(data, "graph_superpixels.pt")
print("✅ Graphe GCN sauvegardé sous 'graph_superpixels.pt' (nodes:", x.shape[0], ")")


✅ Graphe GCN sauvegardé sous 'graph_superpixels.pt' (nodes: 900 )
